# Mentor de Carreira com IA — Colab + OpenRouter

Notebook prático sobre ambiente, API, proteção de chaves, system prompt/guardrails e tokens.

## 1. Preparando o ambiente
Este notebook usa `requests` para chamar a API do OpenRouter.

In [ ]:
!pip -q install requests

## 2. Chave de API com segurança
No Colab, abra **Secrets** (ícone de chave), adicione `OPENROUTER_API_KEY` e habilite o acesso para este notebook. Nunca cole a chave diretamente no código.

In [ ]:
from google.colab import userdata
import requests

API_KEY = userdata.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("Configure OPENROUTER_API_KEY nos Secrets do Colab.")
print("Chave carregada (não exibida).")

## 3. Conectando e listando modelos
A lista pode mudar. O sufixo `:free` identifica opções gratuitas no catálogo, mas confirme os limites e disponibilidade atuais.

In [ ]:
BASE_URL = "https://openrouter.ai/api/v1"
headers = {"Authorization": f"Bearer {API_KEY}"}
resp = requests.get(f"{BASE_URL}/models", headers=headers, timeout=30)
resp.raise_for_status()
models = resp.json().get("data", [])
free_models = [m for m in models if ":free" in m.get("id", "")]
print("Modelos disponíveis:", len(models), "| modelos :free:", len(free_models))
for model in free_models[:25]:
    print(model.get("id"))

## 4. System prompt e guardrails
Estas instruções orientam o mentor. Elas ajudam, mas não garantem segurança ou precisão absoluta.

In [ ]:
SYSTEM_PROMPT = """Você é um mentor de carreira acolhedor, prático e realista.
Ajude a explorar interesses, competências e caminhos profissionais.
Não prometa emprego, salário, promoção ou sucesso. Não invente fatos, vagas ou dados;
avise quando algo precisar ser verificado. Evite solicitar dados sensíveis.
Explique termos para iniciantes e transforme objetivos em passos concretos.
Quando útil, organize a resposta em: resumo, 2-4 opções, próximo passo,
e uma pergunta para avançar. Não substitua aconselhamento profissional especializado."""
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
print("System prompt configurado.")

## 5. Primeiro pedido ao modelo
Copie um ID de modelo da lista anterior e substitua o placeholder abaixo. Para testes sem cobrança, selecione um modelo `:free` disponível.

In [ ]:
MODEL = "COLE_AQUI_O_ID_DO_MODELO"

def ask_ai(prompt, model=MODEL, history=None):
    if model == "COLE_AQUI_O_ID_DO_MODELO":
        raise ValueError("Configure MODEL com um ID válido da seção 3.")
    conversation = list(history or messages)
    conversation.append({"role": "user", "content": prompt})
    payload = {"model": model, "messages": conversation, "temperature": 0.5}
    response = requests.post(
        f"{BASE_URL}/chat/completions",
        headers={**headers, "Content-Type": "application/json"},
        json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()
    answer = data["choices"][0]["message"]["content"]
    usage = data.get("usage", {})
    print(answer)
    print("Uso de tokens reportado pela API:", usage)
    new_history = conversation + [{"role": "assistant", "content": answer}]
    return answer, usage, new_history

# Exemplo: answer, usage, history = ask_ai("Como descubro áreas profissionais que combinam comigo?")

## 6. Chat com o mentor
Execute a célula abaixo para conversar; digite `sair` para encerrar.

In [ ]:
chat_history = list(messages)
while True:
    user_input = input("Digite sua pergunta (ou sair): ").strip()
    if user_input.lower() in {"sair", "exit", "quit"}:
        print("Sessão encerrada.")
        break
    if not user_input:
        continue
    answer, usage, chat_history = ask_ai(user_input, history=chat_history)
    chat_history.append({"role": "user", "content": user_input})

## 7. Tokens, privacidade e desafio
A API pode retornar tokens de entrada/saída e total em `usage`. A cobrança, limites e métricas variam por modelo e podem mudar.

**Desafio:** (1) pergunte como mudar de área sem experiência; (2) solicite um plano de 30 dias; (3) ajuste o prompt para respostas mais concisas; (4) compare duas respostas; (5) confira o consumo e os preços/limites do modelo.

**O que aprendemos:** Python conecta à IA via API; Secrets protege a chave; system prompts orientam comportamento; guardrails são camadas de mitigação, não garantias; tokens medem o processamento e podem influenciar custo. Evite inserir dados pessoais/confidenciais.